# Bias, Variance & Regularization — Student Lab (Breast Cancer)

You will diagnose under/overfitting using capacity sweeps and learning curves, then connect that to regularization knobs.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

In [2]:
data = load_breast_cancer()
X = data.data
y = data.target

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
check('shapes', Xtr.shape[0]==ytr.shape[0] and Xva.shape[0]==yva.shape[0])
print('base_rate', float(y.mean()))

OK: shapes
base_rate 0.6274165202108963


## Section 1 — Baseline

### Task 1.1: Logistic regression baseline

# TODO:
- Fit a logistic regression with standardization
- Report train and val accuracy

**Checkpoint:** Why is logistic regression usually lower-variance than deep trees?

In [3]:
# TODO
lr = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
lr.fit(Xtr, ytr)
print('train_acc', accuracy_score(ytr, lr.predict(Xtr)))
print('val_acc', accuracy_score(yva, lr.predict(Xva)))

# Deep trees overfit the data strongly (deeper the tree, more overfitting there is). Logistic regression cannot capture relations more complex than linear relations and hence unlikely to overfit to any data that is anything beyond linear

train_acc 0.992462311557789
val_acc 0.9590643274853801


## Section 2 — Capacity sweep (Decision Tree depth)

### Task 2.1: Depth sweep

# TODO:
- For depths = 1..15, fit a DecisionTreeClassifier(max_depth=d)
- Track train_acc and val_acc
- Print the best depth by val accuracy

**Checkpoint:** Which depths look high-bias? Which look high-variance?

In [4]:
depths = list(range(1, 16))
train_acc = []
val_acc = []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=0)
    clf.fit(Xtr, ytr)
    train_acc.append(accuracy_score(ytr, clf.predict(Xtr)))
    val_acc.append(accuracy_score(yva, clf.predict(Xva)))

best_i = int(np.argmax(val_acc))
print('best_depth', depths[best_i], 'val_acc', val_acc[best_i])
print('depth, train, val (first 5)')
for i in range(len(depths)):
    print(depths[i], train_acc[i], val_acc[i])

# Large depths have high varianace because they overfit on the data ( higher training accuracy).
# Shallower depths have higher bias because they do not capture some relations entirely and there will be error in that direction consistently across all predictions (lower training accuracy)

best_depth 5 val_acc 0.9122807017543859
depth, train, val (first 5)
1 0.9321608040201005 0.8888888888888888
2 0.9422110552763819 0.9064327485380117
3 0.9798994974874372 0.9005847953216374
4 0.9899497487437185 0.9064327485380117
5 0.9974874371859297 0.9122807017543859
6 1.0 0.9064327485380117
7 1.0 0.9064327485380117
8 1.0 0.9064327485380117
9 1.0 0.9064327485380117
10 1.0 0.9064327485380117
11 1.0 0.9064327485380117
12 1.0 0.9064327485380117
13 1.0 0.9064327485380117
14 1.0 0.9064327485380117
15 1.0 0.9064327485380117


## Section 3 — Regularization sweep (LogReg C)

### Task 3.1: Sweep C for L2 logistic regression

# TODO:
- Try C in [0.01, 0.1, 1, 10, 100]
- Compare train vs val accuracy

**Checkpoint:** What does decreasing C do to bias/variance?

In [8]:
Cs = [0.01, 0.1, 1.0, 10.0, 100.0]
for C in Cs:
    lrC = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, C=C, penalty='l2'))
    ])
    lrC.fit(Xtr, ytr)
    tr = accuracy_score(ytr, lrC.predict(Xtr))
    va = accuracy_score(yva, lrC.predict(Xva))
    print('C', C, 'train', tr, 'val', va)

# Increasing C means reducing the regression component (making the data loss more important).
# So as we increae C, the model overfits more and validation accuracy does not increase much more beyond certain value of C

C 0.01 train 0.957286432160804 val 0.9415204678362573
C 0.1 train 0.9899497487437185 val 0.9590643274853801
C 1.0 train 0.992462311557789 val 0.9590643274853801
C 10.0 train 0.992462311557789 val 0.9532163742690059
C 100.0 train 0.9974874371859297 val 0.9590643274853801


### Task 3.2: L1 vs L2 sparsity

# TODO:
- Fit L1 logistic regression (solver=liblinear)
- Count how many coefficients are exactly 0
- Compare to L2

**FAANG Gotcha:** correlated features can make L1 unstable (it picks one arbitrarily).

In [21]:
# L2 model
lr_l2 = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, C=1.0, penalty='l2'))])
lr_l2.fit(Xtr, ytr)
w_l2 = lr_l2.named_steps['model'].coef_.ravel()

# TODO: L1 model
lr_l1 = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, C=1.0, penalty='l1',solver='liblinear'))])
lr_l1.fit(Xtr, ytr)
w_l1 = lr_l1.named_steps['model'].coef_.ravel()

print('l2 zeros', int(np.sum(np.isclose(w_l2, 0.0))))
print('l1 zeros', int(np.sum(np.isclose(w_l1, 0.0))))

l2 zeros 0
l1 zeros 18


## Section 4 — Learning curve by subsampling

### Task 4.1: Train with increasing data sizes

# TODO:
- For fractions in [0.1, 0.2, 0.4, 0.6, 0.8, 1.0], train logistic regression on that fraction
- Track train vs val accuracy

**Interview Angle:** How do you decide whether to get more data vs regularize more?

In [23]:
fracs = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
idx_all = np.arange(len(Xtr))

for f in fracs:
    k = max(5, int(f * len(Xtr)))
    idx = rng.choice(idx_all, size=k, replace=False)
    Xsub, ysub = Xtr[idx], ytr[idx]
    lr_sub = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, C=1.0))])
    lr_sub.fit(Xsub, ysub)
    tr = accuracy_score(ysub, lr_sub.predict(Xsub))
    va = accuracy_score(yva, lr_sub.predict(Xva))
    print('frac', f, 'n', k, 'train', tr, 'val', va)


# If we have low training and validation accuracy -> then it means model is not capable of learning the complexity of the data -> model is too simple, so reduce regularization
# If we have high training and low validation accuracy, then it means model has overfit. So adding data or regularization, both help
# Generally, we start with regularization as gathering data might be expensive in the real world. But if regularization does not help, then we try to add more data

frac 0.1 n 39 train 1.0 val 0.9473684210526315
frac 0.2 n 79 train 0.9873417721518988 val 0.935672514619883
frac 0.4 n 159 train 1.0 val 0.9473684210526315
frac 0.6 n 238 train 0.9831932773109243 val 0.9473684210526315
frac 0.8 n 318 train 0.9937106918238994 val 0.9590643274853801
frac 1.0 n 398 train 0.992462311557789 val 0.9590643274853801


---
## Submission Checklist
- Baseline LR results
- Depth sweep table + bias/variance diagnosis
- C sweep interpretation
- L1 vs L2 sparsity counts
- Learning curve reasoning
